# Import Environment variables

In [1]:
%run /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Setting_Env_Variables.ipynb

Found bucket: id=rw-migration-aou-rw-f7a4d148, bucketName=rw-migration-aou-rw-f7a4d148
-> Assigned migration variables (ID: rw-migration-aou-rw-f7a4d148)
Found bucket: id=temporary-workspace-bucket, bucketName=temporary-workspace-bucket-wb-perky-cabbage-8342
Found bucket: id=workspace-bucket, bucketName=workspace-bucket-wb-perky-cabbage-8342
✅ Successfully identified latest dataset: wb-silky-artichoke-2408.C2024Q3R9

Variables extracted:
GOOGLE_CLOUD_PROJECT: wb-perky-cabbage-8342
WORKSPACE_BUCKET: gs://workspace-bucket-wb-perky-cabbage-8342
WORKSPACE_TEMP_BUCKET: gs://temporary-workspace-bucket-wb-perky-cabbage-8342
WORKSPACE_CDR: wb-silky-artichoke-2408.C2024Q3R9
bucket_aou_tutorial: NOT FOUND
bucket_id_aou_tutorial: NOT FOUND
bucket_migrated: gs://rw-migration-aou-rw-f7a4d148
bucket_id_migrated: rw-migration-aou-rw-f7a4d148

✅ Saved to /home/jupyter/.bashrc
C2024Q3R9 BQ_DATASET
Multi-trait-GWAS-in-admixed-populations GIT_REPO
dataset_test2 BQ_DATASET
prep_C2024Q3R9 BQ_DATASET
rw-mig

In [2]:
%run /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Setting_Env_Variables_p2.ipynb

WORKSPACE_CDR = wb-silky-artichoke-2408.C2024Q3R9
WORKSPACE_BUCKET = gs://workspace-bucket-wb-perky-cabbage-8342
GOOGLE_PROJECT = wb-perky-cabbage-8342
Done! 10 variables saved to: /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Setting_Env_Variables_p2.R
Done! 10 variables saved to: /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Setting_Env_Variables.sas


# Selection des SNPs du PRS313 version b38

## Download srNGS pvar for each chromosome from 1 to 22

In [ ]:
%%bash

BASE_URI="gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/pgen"
for chr in {1..22}; do
    echo "Downloading chr${chr}..."
    gsutil -u $GOOGLE_PROJECT cp \
        ${BASE_URI}/acaf_threshold.chr${chr}.pvar \
        srNGS/
done

## Copy prs313 list to bucket

In [4]:
name_of_file_in_bucket = "prs313_b38_all_v2.csv"

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file from the bucket to the current working space
os.system(f"gsutil cp 'Datas/{name_of_file_in_bucket}' '{my_bucket}/Data'")

print(f'[INFO] {name_of_file_in_bucket} is successfully uploaded into your working space')

Copying file://Datas/prs313_b38_all_v2.csv [Content-Type=text/csv]...
/ [1 files][  7.3 KiB/  7.3 KiB]                                                
Operation completed over 1 objects/7.3 KiB.                                      


[INFO] prs313_b38_all_v2.csv is successfully uploaded into your working space


## Try to find markers 

In [5]:
from pathlib import Path
import pandas as pd

# 1. Charger et nettoyer le fichier d'intervalles BED (prs313_b38_all.csv)
file_path = "Datas/prs313_b38_all_v2.csv"

# Lecture du fichier séparé par des points-virgules (;)
snps = pd.read_csv(file_path, sep="\t", header=None, dtype=str)

# Si une colonne d'index/numérotation est présente au début, on garde les 3 dernières colonnes
if snps.shape[1] > 3:
    snps = snps.iloc[:, -3:]

snps.columns = ["chr", "start", "end"]

# Nettoyage des chaînes : suppression des espaces et extraction du numéro de chromosome
snps["chr_clean"] = snps["chr"].str.replace("chr", "", case=False).str.strip()
snps["pos_clean"] = snps["end"].str.strip()

# Création du set des paires (chr, pos) cibles
target_pairs = set(zip(snps["chr_clean"], snps["pos_clean"]))

print(f"[INFO] Nombre de positions cibles uniques : {len(target_pairs)}")

# 2. Scanner les fichiers .pvar
matches = []

for n in range(1, 23):
    pvar_path = Path(f"srNGS/acaf_threshold.chr{n}.pvar")
    if not pvar_path.exists():
        print(f"[WARN] Fichier manquant : {pvar_path}")
        continue

    with pvar_path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.startswith("#"):
                continue

            parts = line.strip().split()
            if len(parts) >= 5:
                chrom = parts[0].replace("chr", "").strip()
                pos = parts[1].strip()

                if (chrom, pos) in target_pairs:
                    matches.append(
                        {
                            "chr_pvar": chrom,
                            "pos_pvar": pos,
                            "id_pvar": parts[2],
                            "ref_pvar": parts[3],
                            "alt_pvar": parts[4],
                            "full_pvar_line": line.strip(),
                        }
                    )

df_matches = pd.DataFrame(matches)
print(f"[SUCCESS] Total de variants pvar retrouvés : {len(df_matches)}")

# 3. Fusion avec le DataFrame initial pour repérer les variants manquants
df_merged = snps.merge(
    df_matches,
    left_on=["chr_clean", "pos_clean"],
    right_on=["chr_pvar", "pos_pvar"],
    how="left",
)

# Afficher les variants absents
missing = df_merged[df_merged["id_pvar"].isna()]
print(
    f"\n[INFO] {len(missing)} variants n'ont pas été trouvés dans acaf_threshold :"
)
print(missing[["chr", "start", "end"]])

# Export du résultat final
df_merged.to_csv("Datas/snps_vs_pvar_python_join.csv", index=False)

[INFO] Nombre de positions cibles uniques : 311
[SUCCESS] Total de variants pvar retrouvés : 311

[INFO] 0 variants n'ont pas été trouvés dans acaf_threshold :
Empty DataFrame
Columns: [chr, start, end]
Index: []


In [6]:
!grep 145829794  srNGS/acaf_threshold.chr1.pvar

chr1	145829794	.	G	A,T	.	AC=246302,1;AF=0.297,1.205e-06;AN=829608;AS_QUALapprox=0|21738947|79;CALIBRATION_SENSITIVITY=0.4914,0.9961;QUALapprox=85;SCORE=-0.3497,-0.6765


## Data export

# Extraction des 313 variants sur les données srNGS

In [ ]:
%%bash

BASE_URI="gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/pgen"
DATAS_FOLDER="/home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Datas"
SRNGS_FILE="/home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/srNGS"
SRNGS_PRS313_FILE="/home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/srNGS_prs313"
mkdir -p srNGS_prs313

for chr in {1..22}; do
    echo "=== Traitement du Chromosome ${chr} ==="
    # 1. Téléchargement des 2 fichiers PLINK2 restants pour le chromosome
    gsutil -u $GOOGLE_PROJECT -m -o GSUtil:show_progress_bar=True cp ${BASE_URI}/acaf_threshold.chr${chr}.pgen ${SRNGS_FILE}/
    gsutil -u $GOOGLE_PROJECT -m -o GSUtil:show_progress_bar=True cp ${BASE_URI}/acaf_threshold.chr${chr}.psam ${SRNGS_FILE}/
    
    # 2. Extraction ciblée des 313 variants avec PLINK2
    plink2 --pfile ${SRNGS_FILE}/acaf_threshold.chr${chr} --extract bed1 ${DATAS_FOLDER}/prs313_b38_all_v2.csv --make-pgen --out ${SRNGS_PRS313_FILE}/prs313_all_chr${chr}
    
    # 3. Nettoyage immédiat du gros fichier .pgen brut pour économiser l'espace disque
    rm ${SRNGS_FILE}/acaf_threshold.chr${chr}.pgen
done

echo "[OK] Extraction terminée pour tous les chromosomes !"

## Fusion des fichiers en un seul `.bcf`

In [4]:
%%bash

SRNGS_PRS313_FILE="/home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/srNGS_prs313"

ls -v ${SRNGS_PRS313_FILE}/prs313_all_chr*.pgen | sed 's/.pgen//g' > ${SRNGS_PRS313_FILE}/list_concat_chrs.txt
plink2 --pmerge-list ${SRNGS_PRS313_FILE}/list_concat_chrs.txt --export bcf id-paste=iid --out ${SRNGS_PRS313_FILE}/prs313_all
bcftools index ${SRNGS_PRS313_FILE}/prs313_all.bcf

PLINK v2.0.0-a.6.9LM 64-bit Intel (29 Jan 2025)    cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/srNGS_prs313/prs313_all.log.
Options in effect:
  --export bcf id-paste=iid
  --out /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/srNGS_prs313/prs313_all
  --pmerge-list /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/srNGS_prs313/list_concat_chrs.txt

Start time: Sat Aug 29 20:16:33 2026
26036 MiB RAM detected, ~24877 available; reserving 13018 MiB for main
workspace.
Using up to 4 compute threads.
--pmerge-list: 22 filesets specified.
--pmerge-list: 414830 samples present.
--pmerge-list: Merged .psam written to
/home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/srNGS_prs313/prs313_all.psam
.
--pmerge-list: 22 .pvar files scanned, headers merged.
Concatenation job detected.
Concatenatin

In [7]:
import os
import subprocess

destination_filename = 'prs313_all.bcf'

# Récupère le nom du bucket Google Cloud depuis la variable d'environnement
my_bucket = os.getenv('WORKSPACE_BUCKET')

args = ["gsutil", "cp", f"srNGS_prs313/{destination_filename}", f"{my_bucket}/Data/"]
output = subprocess.run(args, capture_output=True)

# Affiche les éventuelles erreurs retournées par gsutil
output.stderr

b'Copying file://srNGS_prs313/prs313_all.bcf [Content-Type=application/octet-stream]...\n/ [0 files][    0.0 B/ 32.8 MiB]                                                \r/ [1 files][ 32.8 MiB/ 32.8 MiB]                                                \r-\r\nOperation completed over 1 objects/32.8 MiB.                                     \n'

# Cohorte Cancer - Femme (Controlled Tier Dataset v8)

## With Breast Cancer (BC)

In [1]:
# This snippet assumes that you run setup first

# This code lists objects in your Google Bucket

import os
import subprocess

# Get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# List objects in the bucket
print(subprocess.check_output(f"gsutil ls -r {my_bucket}", shell=True).decode('utf-8'))

gs://fc-secure-4f907dc3-1aa1-4aaa-8566-86d601589221/Data/:
gs://fc-secure-4f907dc3-1aa1-4aaa-8566-86d601589221/Data/
gs://fc-secure-4f907dc3-1aa1-4aaa-8566-86d601589221/Data/aou_admixture_estimates_rye_v8.Q
gs://fc-secure-4f907dc3-1aa1-4aaa-8566-86d601589221/Data/df_ancestry_final.tsv
gs://fc-secure-4f907dc3-1aa1-4aaa-8566-86d601589221/Data/df_final_cohort.tsv
gs://fc-secure-4f907dc3-1aa1-4aaa-8566-86d601589221/Data/df_not_breast_cancer_final.tsv
gs://fc-secure-4f907dc3-1aa1-4aaa-8566-86d601589221/Data/df_plink_fam_files_array.tsv
gs://fc-secure-4f907dc3-1aa1-4aaa-8566-86d601589221/Data/df_relatedness_final.tsv
gs://fc-secure-4f907dc3-1aa1-4aaa-8566-86d601589221/Data/df_rye_final.tsv
gs://fc-secure-4f907dc3-1aa1-4aaa-8566-86d601589221/Data/df_with_breast_cancer_final.tsv
gs://fc-secure-4f907dc3-1aa1-4aaa-8566-86d601589221/Data/pheno_plink.tsv
gs://fc-secure-4f907dc3-1aa1-4aaa-8566-86d601589221/Data/prs313_b38.csv
gs://fc-secure-4f907dc3-1aa1-4aaa-8566-86d601589221/Data/relatedness_flag

In [2]:
import pandas as pd

name_of_file_in_bucket = "df_final_cohort.tsv"

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file from the bucket to the current working space
os.system(f"gsutil cp '{my_bucket}/Data/{name_of_file_in_bucket}' .")

print(f'[INFO] {name_of_file_in_bucket} is successfully downloaded into your working space')
# save dataframe in a csv file in the same workspace as the notebook
df_final_cohort = pd.read_csv(name_of_file_in_bucket)

Copying gs://fc-secure-4f907dc3-1aa1-4aaa-8566-86d601589221/Data/df_final_cohort.tsv...
\ [1 files][117.0 MiB/117.0 MiB]                                                
Operation completed over 1 objects/117.0 MiB.                                    


[INFO] df_final_cohort.tsv is successfully downloaded into your working space


In [3]:
df_final_cohort["has_BC"].value_counts()

has_BC
0    256793
1     10371
Name: count, dtype: int64

# European cohort

## Selection of samples

In [10]:
df_rye = pd.read_csv("aou_admixture_estimates_rye_v8.Q", sep="\t")

In [11]:
list_eur = df_rye[df_rye["eur"] >= 0.8]["research_id"].to_list()

In [12]:
df_eur = df_final_cohort[df_final_cohort["B"].isin(list_eur)].copy()

In [13]:
df_eur["#IID"] = df_eur["B"]

In [14]:
keep_ids_eur = df_eur[["#IID","B"]].rename(columns={"B":"IID"})

In [15]:
keep_ids_eur.to_csv("keep_ids_eur.txt", sep="\t", index=False)

In [16]:
pheno_bc_eur = df_eur[['B', 'has_BC']].rename(columns={"B":"#IID"})

In [17]:
pheno_bc_eur.value_counts('has_BC')

has_BC
0    106109
1      5290
Name: count, dtype: int64

In [18]:
pheno_bc_eur.head()

,#IID,has_BC
4,1000070,0
5,1000091,0
7,1000095,0
12,1000127,0
25,1000262,0


In [19]:
pheno_bc_eur.to_csv("pheno_bc_eur.csv", sep="\t", index=False)

## Processing of genetic data

In [20]:
# Selection of european sample for chr10
!plink2 --pfile acaf_threshold.chr10 --keep keep_ids_eur.txt --make-pgen --out filtered_eur_chr10

PLINK v2.0.0-a.6.12LM 64-bit Intel (20 Apr 2025)   cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to filtered_eur_chr10.log.
Options in effect:
  --keep keep_ids_eur.txt
  --make-pgen
  --out filtered_eur_chr10
  --pfile acaf_threshold.chr10

Start time: Tue Oct 14 15:06:34 2025
14993 MiB RAM detected, ~13455 available; reserving 7496 MiB for main
workspace.
Using up to 4 compute threads.
414830 samples (0 females, 0 males, 414830 ambiguous; 414830 founders) loaded
from acaf_threshold.chr10.psam.
2727345 variants loaded from acaf_threshold.chr10.pvar.
Note: No phenotype data present.
--keep: 111399 samples remaining.
111399 samples (0 females, 0 males, 111399 ambiguous; 111399 founders)
remaining after main filters.
Writing filtered_eur_chr10.psam ... done.
Writing filtered_eur_chr10.pvar ... 10101111121213131414151516161717181819202021212222232324242525262627272828292930303131323233333434353536363737383839404041414242

## Obtaining allele frequencies

In [21]:
# CASES
!plink2 \
--pfile filtered_eur_chr10 \
--pheno pheno_bc_eur.csv \
--pheno-name has_BC \
--1 \
--keep-if has_BC == 1 \
--chr 10 --from-bp 121580593 --to-bp 121580593 \
--freq \
--out cases_eur

PLINK v2.0.0-a.6.12LM 64-bit Intel (20 Apr 2025)   cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to cases_eur.log.
Options in effect:
  --1
  --chr 10
  --freq
  --from-bp 121580593
  --keep-if has_BC == 1
  --out cases_eur
  --pfile filtered_eur_chr10
  --pheno pheno_bc_eur.csv
  --pheno-name has_BC
  --to-bp 121580593

Start time: Tue Oct 14 15:29:30 2025
14993 MiB RAM detected, ~13478 available; reserving 7496 MiB for main
workspace.
Using up to 4 compute threads.
111399 samples (0 females, 0 males, 111399 ambiguous; 111399 founders) loaded
from filtered_eur_chr10.psam.
2727345 variants loaded from filtered_eur_chr10.pvar.
1 binary phenotype loaded (5290 cases, 106109 controls).
--keep-if: 106109 samples removed.
5290 samples (0 females, 0 males, 5290 ambiguous; 5290 founders) remaining
after main filters.
5290 cases and 0 controls remaining after main filters.
Calculating allele frequencies... done.
--freq: Allele

In [22]:
# CONTROLS
!plink2 \
--pfile filtered_eur_chr10 \
--pheno pheno_bc_eur.csv \
--pheno-name has_BC \
--1 \
--keep-if has_BC == 0 \
--chr 10 --from-bp 121580593 --to-bp 121580593 \
--freq \
--out control_eur

PLINK v2.0.0-a.6.12LM 64-bit Intel (20 Apr 2025)   cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to control_eur.log.
Options in effect:
  --1
  --chr 10
  --freq
  --from-bp 121580593
  --keep-if has_BC == 0
  --out control_eur
  --pfile filtered_eur_chr10
  --pheno pheno_bc_eur.csv
  --pheno-name has_BC
  --to-bp 121580593

Start time: Tue Oct 14 15:29:37 2025
14993 MiB RAM detected, ~13485 available; reserving 7496 MiB for main
workspace.
Using up to 4 compute threads.
111399 samples (0 females, 0 males, 111399 ambiguous; 111399 founders) loaded
from filtered_eur_chr10.psam.
2727345 variants loaded from filtered_eur_chr10.pvar.
1 binary phenotype loaded (5290 cases, 106109 controls).
--keep-if: 5290 samples removed.
106109 samples (0 females, 0 males, 106109 ambiguous; 106109 founders)
remaining after main filters.
0 cases and 106109 controls remaining after main filters.
Calculating allele frequencies... done.
--fr

## Frequency analysis

In [23]:
pd.read_csv("cases_eur.afreq", sep="\t")

,#CHROM,ID,REF,ALT,ALT_FREQS,OBS_CT
0,10,.,A,G,0.055198,10580


In [25]:
pd.read_csv("control_eur.afreq", sep="\t")

,#CHROM,ID,REF,ALT,ALT_FREQS,OBS_CT
0,10,.,A,G,0.059324,212174


## GWAS

In [26]:
!plink2 \
    --pfile filtered_eur_chr10 \
    --pheno pheno_bc_eur.csv \
    --pheno-name has_BC \
    --1 \
    --glm hide-covar single-prec-cc allow-no-covars \
    --chr 10 --from-bp 121580593 --to-bp 121580593 \
    --threads 8 \
    --out filtered_eur_chr10_gwas

PLINK v2.0.0-a.6.12LM 64-bit Intel (20 Apr 2025)   cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to filtered_eur_chr10_gwas.log.
Options in effect:
  --1
  --chr 10
  --from-bp 121580593
  --glm hide-covar single-prec-cc allow-no-covars
  --out filtered_eur_chr10_gwas
  --pfile filtered_eur_chr10
  --pheno pheno_bc_eur.csv
  --pheno-name has_BC
  --threads 8
  --to-bp 121580593

Start time: Tue Oct 14 15:31:26 2025
14993 MiB RAM detected, ~13479 available; reserving 7496 MiB for main
workspace.
Using up to 8 compute threads.
111399 samples (0 females, 0 males, 111399 ambiguous; 111399 founders) loaded
from filtered_eur_chr10.psam.
2727345 variants loaded from filtered_eur_chr10.pvar.
1 binary phenotype loaded (5290 cases, 106109 controls).
Calculating allele frequencies... done.
1 variant remaining after main filters.
samples.
--glm logistic-Firth hybrid regression on phenotype 'has_BC': done.
Results written to filte

## GWAS Result

In [27]:
pd.read_csv("filtered_eur_chr10_gwas.has_BC.glm.logistic.hybrid", sep="\t")

,#CHROM,POS,ID,REF,ALT,PROVISIONAL_REF?,A1,OMITTED,A1_FREQ,FIRTH?,TEST,OBS_CT,OR,LOG(OR)_SE,Z_STAT,P,ERRCODE
0,10,121580593,.,A,G,N,G,A,0.059128,N,ADD,111377,0.926521,0.043515,-1.75384,0.079458,.


# African cohort

## Selection samples

In [28]:
list_afr = df_rye[df_rye["afr"] >= 0.8]["research_id"].to_list()

In [29]:
df_afr = df_final_cohort[df_final_cohort["B"].isin(list_afr)].copy()

In [30]:
df_afr["#IID"] = df_afr["B"]

In [31]:
keep_ids_afr = df_afr[["#IID","B"]].rename(columns={"B":"IID"})

In [32]:
keep_ids_afr.to_csv("keep_ids_afr.txt", sep="\t", index=False)

In [33]:
pheno_bc_afr = df_afr[['B', 'has_BC']].rename(columns={"B":"#IID"})

In [34]:
pheno_bc_afr.value_counts('has_BC')

has_BC
0    31040
1      826
Name: count, dtype: int64

In [35]:
pheno_bc_afr.head()

,#IID,has_BC
0,1000039,0
13,1000151,0
19,1000195,0
21,1000204,0
26,1000265,0


In [36]:
pheno_bc_afr.to_csv("pheno_bc_afr.csv", sep="\t", index=False)

## Processing of genetic data

In [ ]:
!plink2 --pfile acaf_threshold.chr10 --keep keep_ids_afr.txt --make-pgen --out filtered_afr_chr10

PLINK v2.0.0-a.6.12LM 64-bit Intel (20 Apr 2025)   cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to filtered_afr_chr10.log.
Options in effect:
  --keep keep_ids_afr.txt
  --make-pgen
  --out filtered_afr_chr10
  --pfile acaf_threshold.chr10

Start time: Tue Oct 14 15:31:37 2025
14993 MiB RAM detected, ~13462 available; reserving 7496 MiB for main
workspace.
Using up to 4 compute threads.
414830 samples (0 females, 0 males, 414830 ambiguous; 414830 founders) loaded
from acaf_threshold.chr10.psam.
2727345 variants loaded from acaf_threshold.chr10.pvar.
Note: No phenotype data present.
--keep: 31866 samples remaining.
31866 samples (0 females, 0 males, 31866 ambiguous; 31866 founders) remaining
after main filters.
Writing filtered_afr_chr10.psam ... done.
Writing filtered_afr_chr10.pvar ... 101011111212131314141515161617171818192020212122222323242425252626272728282929303031313232333334343535363637373838394040414142424343

## Obtaining allele frequencies

In [ ]:
!plink2 \
--pfile filtered_afr_chr10 \
--pheno pheno_bc_afr.csv \
--pheno-name has_BC \
--1 \
--keep-if has_BC == 1 \
--chr 10 --from-bp 121580593 --to-bp 121580593 \
--freq \
--out cases_afr

In [ ]:
!plink2 \
--pfile filtered_afr_chr10 \
--pheno pheno_bc_afr.csv \
--pheno-name has_BC \
--1 \
--keep-if has_BC == 0 \
--chr 10 --from-bp 121580593 --to-bp 121580593 \
--freq \
--out control_afr

## Frequency analysis

In [ ]:
pd.read_csv("cases_afr.afreq", sep="\t")

In [ ]:
pd.read_csv("control_afr.afreq", sep="\t")

## GWAS

In [ ]:
!plink2 \
    --pfile filtered_afr_chr10 \
    --pheno pheno_bc_afr.csv \
    --pheno-name has_BC \
    --1 \
    --glm hide-covar single-prec-cc allow-no-covars \
    --chr 10 --from-bp 121580593 --to-bp 121580593 \
    --threads 8 \
    --out filtered_afr_chr10_gwas

## GWAS Result

In [ ]:
pd.read_csv("filtered_afr_chr10_gwas.has_BC.glm.logistic.hybrid", sep="\t")